# Capítulo 4: Separando Dados e Instruções

- [Lição](#licao)
- [Exercícios](#exercicios)
- [Playground de Exemplos](#playground-de-exemplos)

## Configuração

Execute a célula de configuração abaixo para carregar sua API key e definir a função auxiliar `get_completion`.

In [ ]:
import sys, re
sys.path.insert(0, '/workspaces/prompt-eng-ia')
from ollama_helper import get_completion, MODEL_NAME, client

---

## Lição

Muitas vezes, não queremos escrever prompts completos, mas sim **templates de prompt que podem ser modificados posteriormente com dados de entrada adicionais antes de enviar ao Claude**. Isso pode ser útil quando você quer que o Claude faça a mesma coisa sempre, mas os dados que ele usa para a tarefa podem variar a cada vez.

Felizmente, podemos fazer isso de forma bem simples **separando o esqueleto fixo do prompt da entrada variável do usuário e, em seguida, substituindo essa entrada no prompt** antes de enviar o prompt completo ao Claude.

A seguir, veremos passo a passo como escrever um template de prompt com substituição, bem como como inserir a entrada do usuário.

### Exemplos

Neste primeiro exemplo, pedimos ao Claude para agir como um gerador de sons de animais. Observe que o prompt completo enviado ao Claude é apenas o `PROMPT_TEMPLATE` com a entrada substituída (neste caso, "Cow"). Note que a palavra "Cow" substitui o marcador `ANIMAL` via f-string quando imprimimos o prompt completo.

**Nota:** Na prática, você não precisa dar nenhum nome específico à sua variável de marcador. Chamamos de `ANIMAL` neste exemplo, mas poderíamos facilmente tê-la chamado de `CREATURE` ou `A` (embora seja geralmente bom que os nomes de variáveis sejam específicos e relevantes, para que o template do prompt seja fácil de entender mesmo sem a substituição, por questões de legibilidade). Apenas certifique-se de que o nome da variável que você usar seja o mesmo usado na f-string do template do prompt.

In [ ]:
# Conteúdo variável
ANIMAL = "Cow"

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

Por que queremos separar e substituir entradas dessa forma? Bem, **templates de prompt simplificam tarefas repetitivas**. Imagine que você cria uma estrutura de prompt que convida usuários terceiros a enviar conteúdo ao prompt (neste caso, o animal cujo som eles querem gerar). Esses usuários não precisam escrever nem ver o prompt completo. Tudo o que precisam fazer é preencher as variáveis.

Fazemos essa substituição aqui usando variáveis e f-strings, mas você também pode fazer isso com o método format().

**Nota:** Templates de prompt podem ter quantas variáveis forem necessárias!

Ao introduzir variáveis de substituição dessa forma, é muito importante **garantir que o Claude saiba onde as variáveis começam e terminam** (em relação às instruções ou descrições de tarefas). Vejamos um exemplo em que não há separação entre as instruções e a variável de substituição.

Para os nossos olhos humanos, fica muito claro onde a variável começa e termina no template do prompt abaixo. No entanto, no prompt totalmente substituído, essa delimitação se torna imprecisa.

In [ ]:
# Conteúdo variável
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

Aqui, **o Claude acha que "Yo Claude" faz parte do e-mail que ele deve reescrever**! Você pode perceber isso porque ele começa sua reescrita com "Dear Claude". Para o olho humano, fica claro — especialmente no template do prompt — onde o e-mail começa e termina, mas isso se torna bem menos claro no prompt após a substituição.

Como resolvemos isso? **Envolva a entrada em tags XML**! Fizemos isso abaixo, e como você pode ver, não há mais "Dear Claude" na saída.

[Tags XML](https://docs.anthropic.com/claude/docs/use-xml-tags) são tags entre colchetes angulares como `<tag></tag>`. Elas vêm em pares e consistem em uma tag de abertura, como `<tag>`, e uma tag de fechamento marcada com `/`, como `</tag>`. As tags XML são usadas para envolver conteúdo, assim: `<tag>conteúdo</tag>`.

**Nota:** Embora o Claude possa reconhecer e trabalhar com uma ampla variedade de separadores e delimitadores, recomendamos que você **use especificamente tags XML como separadores** para o Claude, pois ele foi treinado especificamente para reconhecer tags XML como mecanismo de organização de prompts. Fora do function calling, **não existem tags XML especiais com as quais o Claude foi treinado que você deva usar para maximizar seu desempenho**. Intencionalmente tornamos o Claude bastante maleável e personalizável dessa forma.

In [ ]:
# Conteúdo variável
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

Vejamos outro exemplo de como as tags XML podem nos ajudar.

No prompt a seguir, **o Claude interpreta incorretamente qual parte do prompt é a instrução e qual é a entrada**. Ele considera incorretamente `Each is about an animal, like rabbits` como parte da lista devido à formatação, quando o usuário (quem preencheu a variável `SENTENCES`) presumivelmente não queria isso.

In [ ]:
# Conteúdo variável
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

Para corrigir isso, basta **envolver as frases de entrada do usuário em tags XML**. Isso mostra ao Claude onde os dados de entrada começam e terminam, apesar do hífen enganoso antes de `Each is about an animal, like rabbits.`

In [ ]:
# Conteúdo variável
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

**Nota:** Na versão incorreta do prompt "Each is about an animal", precisamos incluir o hífen para que o Claude respondesse de forma incorreta do jeito que queríamos para este exemplo. Isso é uma lição importante sobre prompting: **detalhes pequenos importam**! Sempre vale a pena **revisar seus prompts em busca de erros de digitação e gramaticais**. O Claude é sensível a padrões (em seus primeiros anos, antes do fine-tuning, era uma ferramenta bruta de previsão de texto), e é mais propenso a cometer erros quando você comete erros, mais inteligente quando você soa inteligente, mais bobo quando você soa bobo, e assim por diante.

Se quiser experimentar os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook para visitar o [**Playground de Exemplos**](#playground-de-exemplos).

---

## Exercícios
- [Exercício 4.1 - Tópico de Haiku](#exercicio-41---topico-de-haiku)
- [Exercício 4.2 - Pergunta sobre Cães com Erros de Digitação](#exercicio-42---pergunta-sobre-caes-com-erros-de-digitacao)
- [Exercício 4.3 - Pergunta sobre Cães Parte 2](#exercicio-43---pergunta-sobre-caes-parte-2)

### Exercício 4.1 - Tópico de Haiku
Modifique o `PROMPT` para que seja um template que receba uma variável chamada `TOPIC` e produza um haiku sobre o tópico. Este exercício tem como objetivo testar sua compreensão da estrutura de templates de variáveis com f-strings.

In [ ]:
# Conteúdo variável
TOPIC = "Pigs"

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f""

# Obtém a resposta do Claude
response = get_completion(PROMPT)

# Função para avaliar a correção do exercício
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(response)
print("\n------------------------------------------ AVALIAÇÃO ------------------------------------------")
print("Este exercício foi resolvido corretamente:", grade_exercise(response))

❓ Se quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_4_1_hint; print(exercise_4_1_hint)

### Exercício 4.2 - Pergunta sobre Cães com Erros de Digitação
Corrija o `PROMPT` adicionando tags XML para que o Claude produza a resposta correta.

Tente não alterar mais nada no prompt. A escrita confusa e cheia de erros é intencional, para que você possa ver como o Claude reage a esses erros.

In [ ]:
# Conteúdo variável
QUESTION = "ar cn brown?"

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Obtém a resposta do Claude
response = get_completion(PROMPT)

# Função para avaliar a correção do exercício
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(response)
print("\n------------------------------------------ AVALIAÇÃO ------------------------------------------")
print("Este exercício foi resolvido corretamente:", grade_exercise(response))

❓ Se quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_4_2_hint; print(exercise_4_2_hint)

### Exercício 4.3 - Pergunta sobre Cães Parte 2
Corrija o `PROMPT` **SEM** adicionar tags XML. Em vez disso, remova apenas uma ou duas palavras do prompt.

Assim como nos exercícios anteriores, tente não alterar mais nada no prompt. Isso mostrará que tipo de linguagem o Claude consegue analisar e compreender.

In [ ]:
# Conteúdo variável
QUESTION = "ar cn brown?"

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Obtém a resposta do Claude
response = get_completion(PROMPT)

# Função para avaliar a correção do exercício
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(response)
print("\n------------------------------------------ AVALIAÇÃO ------------------------------------------")
print("Este exercício foi resolvido corretamente:", grade_exercise(response))

❓ Se quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_4_3_hint; print(exercise_4_3_hint)

### Parabéns!

Se você resolveu todos os exercícios até este ponto, está pronto para avançar para o próximo capítulo. Boas práticas de prompt!

---

## Playground de Exemplos

Esta é uma área para você experimentar livremente os exemplos de prompt apresentados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
# Conteúdo variável
ANIMAL = "Cow"

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Conteúdo variável
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Conteúdo variável
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Conteúdo variável
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Conteúdo variável
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Template de prompt com um marcador para o conteúdo variável
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Imprime a resposta do Claude
print("--------------------------- Prompt completo com substituições de variáveis ---------------------------")
print(PROMPT)
print("\n------------------------------------- Resposta do Claude -------------------------------------")
print(get_completion(PROMPT))